# 01 - EDA - Predicción de reingreso hospitalario en pacientes diabéticos

EDA sobre el dataset de UCI. No se modifica el CSV crudo, todo lo reutilizable
queda en `src/data/` y `src/eda/`.

In [ ]:
# =====================================================================
# Setup — funciona tanto en local (VS Code/Jupyter) como en Google Colab
# =====================================================================
import importlib
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/mzc07/diabetes-readmission-predictor.git"
    REPO_BRANCH = "develop"
    REPO_DIR = Path("/content/diabetes-readmission-predictor")

    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

    # Nos ubicamos en notebooks/ para que las rutas relativas del resto del
    # notebook (../data/raw/..., ../reports/figures) funcionen igual que en local.
    os.chdir(REPO_DIR / "notebooks")

    %pip install -q numpy pandas scikit-learn matplotlib seaborn

%load_ext autoreload
%autoreload 2

sys.path.insert(0, str(Path.cwd().parent))  # para poder importar src/ desde notebooks/

import src.data.loader
import src.data.quality
import src.eda.univariate
import src.eda.bivariate
import src.eda.relevance

# Force reload modules
importlib.reload(src.data.loader)
importlib.reload(src.data.quality)
importlib.reload(src.eda.univariate)
importlib.reload(src.eda.bivariate)
importlib.reload(src.eda.relevance)

# Los módulos se importaban pero las funciones se usaban sin prefijo más abajo
# (habría fallado con NameError). Se traen los nombres explícitamente:
from src.data.loader import load_diabetic_data, load_id_mappings, ID_COLUMNS, validate_schema
from src.data.quality import (
    duplicate_report,
    missing_value_report,
    sentinel_value_scan,
    numeric_out_of_range_report,
)
from src.eda.univariate import target_distribution, numeric_summary, categorical_summary
from src.eda.bivariate import (
    correlation_matrix,
    high_correlation_pairs,
    categorical_vs_target_chi2,
    numeric_vs_target_test,
)
from src.eda.relevance import mutual_info_ranking, cramers_v_ranking

sns.set_theme(style="whitegrid")
FIGURES_DIR = Path("../reports/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)


## Fase 0 — Preparación

In [ ]:
df = load_diabetic_data("../data/raw/diabetic_data.csv")
mappings = load_id_mappings("../data/raw/IDS_mapping.csv")

print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
df.dtypes.value_counts()

**Nota:** `admission_type_id`, `discharge_disposition_id` y `admission_source_id`
son categoricas codificadas como enteros, su significado esta en `IDS_mapping.csv`.
Por eso las mostramos decodificadas.

In [ ]:
for name, table in mappings.items():
    print(f"--- {name} ---")
    display(table)

In [ ]:
problems = validate_schema(df)
if problems:
    for p in problems:
        print("- ", p)
else:
    print("Esquema validado sin problemas.")

**Duplicados y llaves.** `encounter_id` debe ser unico (una fila = un encuentro).
`patient_nbr` no debe serlo, un mismo paciente puede tener varias hospitalizaciones,
por eso el split se hace agrupado por paciente.

In [ ]:
dup_report = duplicate_report(df, key_columns=ID_COLUMNS)
dup_report

## Fase 1 — Calidad de datos

### Valores faltantes

In [ ]:
missing = missing_value_report(df)
missing[missing["pct_faltante"] > 0]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
faltantes = missing[missing["pct_faltante"] > 0]["pct_faltante"]
faltantes.plot(kind="barh", ax=ax, color="#c62828")
ax.set_xlabel("% faltante")
ax.set_title("Porcentaje de valores faltantes por columna")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_missing_values.png", dpi=150)
plt.show()

**Lectura:** `weight` (~97%) es candidata a eliminar. `medical_specialty` (~49%) y
`payer_code` (~40%) estan en un rango medio, se podria evaluar una columna
binaria de "es nulo" antes de imputar directamente.

### Valores centinela

El loader ya convierte los centinelas conocidos (`?`, `Not Available`, `NULL`, `Not Mapped`)
en NaN. Este escaneo es solo para revisar si queda algo suelto.

In [ ]:
sentinel_value_scan(df)

### Consistencia de tipos y outliers

In [ ]:
rangos_esperados = {
    "time_in_hospital": (1, 14),
    "num_lab_procedures": (0, 150),
    "num_procedures": (0, 10),
    "num_medications": (0, 100),
    "number_outpatient": (0, 50),
    "number_emergency": (0, 50),
    "number_inpatient": (0, 30),
    "number_diagnoses": (1, 20),
}
numeric_out_of_range_report(df, rangos_esperados)

## Fase 2 - Análisis univariado

### Target

Se define el target binario `readmitted_lt30` para usarlo en el resto del análisis.

In [ ]:
df["readmitted_lt30"] = df["readmitted"] == "<30"
target_distribution(df, target_col="readmitted")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df["readmitted"].value_counts().plot(kind="bar", ax=axes[0], color="#4a6cf7")
axes[0].set_title("Target original (3 clases)")
axes[0].set_xlabel("")

df["readmitted_lt30"].value_counts().plot(kind="bar", ax=axes[1], color="#f57c00")
axes[1].set_title("Target binario (readmitted_lt30)")
axes[1].set_xticklabels(["No", "Sí"], rotation=0)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_target_distribution.png", dpi=150)
plt.show()

**Desbalance:** cerca del 11% de los encuentros terminan en reingreso &lt;30 días.

### Variables numéricas clave

In [ ]:
numeric_cols = [
    "time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
    "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses",
]
numeric_summary(df, numeric_cols)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flat, numeric_cols):
    sns.histplot(df[col], kde=True, ax=ax, color="#4a6cf7")
    ax.set_title(col)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "03_numeric_distributions.png", dpi=150)
plt.show()

**Lectura:** `number_outpatient`, `number_emergency` y `number_inpatient` están muy
sesgadas a la derecha (la mayoría en 0), es normal en conteos de eventos raros.

### Variables categóricas

In [ ]:
categorical_cols = [
    "race", "gender", "age", "admission_type_id", "discharge_disposition_id",
    "admission_source_id", "medical_specialty", "max_glu_serum", "A1Cresult",
    "insulin", "change", "diabetesMed",
]
categorical_summary(df, categorical_cols)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
orden_edad = sorted(df["age"].dropna().unique(), key=lambda x: int(x.strip("[)").split("-")[0]))
df["age"].value_counts().reindex(orden_edad).plot(kind="bar", ax=ax, color="#388e3c")
ax.set_title("Distribución por rango de edad")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_age_distribution.png", dpi=150)
plt.show()

## Fase 3 — Análisis bivariado / multivariado

### Numérica vs. numérica

In [ ]:
corr = correlation_matrix(df, numeric_cols, method="spearman")

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlación de Spearman entre variables numéricas")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "05_correlation_heatmap.png", dpi=150)
plt.show()

In [ ]:
high_correlation_pairs(corr, threshold=0.8)

### Categórica vs. target

In [ ]:
chi2_results = categorical_vs_target_chi2(df, categorical_cols, target_col="readmitted")
chi2_results

In [ ]:
top_cat = chi2_results.iloc[0]["columna"]
tabla = pd.crosstab(df[top_cat], df["readmitted"], normalize="index")

fig, ax = plt.subplots(figsize=(9, 5))
tabla.plot(kind="bar", stacked=True, ax=ax, colormap="viridis")
ax.set_title(f"{top_cat} vs. readmitted (proporciones)")
ax.legend(title="readmitted", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "06_top_categorical_vs_target.png", dpi=150)
plt.show()

### Numérica vs. target

In [ ]:
num_vs_target = numeric_vs_target_test(df, numeric_cols, target_binary_col="readmitted_lt30")
num_vs_target

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flat, numeric_cols):
    sns.boxplot(data=df, x="readmitted_lt30", y=col, ax=ax, palette=["#4a6cf7", "#f57c00"])
    ax.set_xticklabels(["No", "Sí"])
    ax.set_title(col)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "07_numeric_vs_target_boxplots.png", dpi=150)
plt.show()

## Fase 4 (parcial) - Ranking de relevancia

Solo la parte estadística (mutual information, Cramér's V). La importancia por
árboles y SHAP se hace después, en el notebook de modelado.

In [ ]:
mi_ranking = mutual_info_ranking(df, numeric_cols, target_binary_col="readmitted_lt30")
mi_ranking

In [ ]:
cv_ranking = cramers_v_ranking(df, categorical_cols, target_col="readmitted")
cv_ranking

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
mi_ranking.set_index("columna")["mutual_info"].plot(kind="barh", ax=axes[0], color="#7b1fa2")
axes[0].set_title("Mutual information (numéricas vs. target)")
axes[0].invert_yaxis()

cv_ranking.set_index("columna")["cramers_v"].plot(kind="barh", ax=axes[1], color="#0075ca")
axes[1].set_title("Cramér's V (categóricas vs. target)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "08_relevance_ranking.png", dpi=150)
plt.show()

## Notas para la siguiente etapa

- Eliminar `weight` (~97% nula).
- Evaluar columna binaria "es_nulo" para `medical_specialty` y `payer_code`.
- Imputar el resto de columnas con nulos residuales.
- Split agrupado por `patient_nbr` (hay pacientes con más de un encuentro).
- Clases desbalanceadas (~11% positivos), usar class_weight balanceado y mirar
  Recall/F1/ROC-AUC en vez de accuracy.
- Revisar pares con correlación > 0.8 antes de usar modelos lineales.
- `diag_1/2/3` quedan pendientes de agrupar por código ICD-9, por eso no se
  analizan aquí como categóricas (tienen demasiadas categorías sin agrupar).
- Ver `mi_ranking` y `cv_ranking` arriba para las variables con más señal.